# Week 1 Exercise — Technical Question Explainer

Takes a technical question (e.g. a snippet of code) and gets two LLMs to explain it:
1. `gpt-4o-mini` (OpenAI) with streaming
2. `llama3.2` (local, via Ollama)

Useful as a personal study tool while working through a course.

## Imports & setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [ ]:
# Constants
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [ ]:
# Load env and validate OpenAI key
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key found — check your .env file.")
elif not api_key.startswith('sk-proj-'):
    print("API key found, but doesn't start with 'sk-proj-' — verify it.")
else:
    print("API key looks good.")

In [ ]:
# Clients — same OpenAI library used for both, just different base URL for Ollama
openai = OpenAI()
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

## The question to ask

Edit the `question` string below to ask anything technical.

In [ ]:
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
system_prompt = """
You are a senior software engineer who explains technical concepts clearly and concisely.
When given a code snippet or question, explain what it does, why it works that way,
and any subtle behavior the reader should be aware of. Respond in markdown.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

## Answer from GPT-4o-mini (streamed)

In [ ]:
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)

## Answer from Llama 3.2 (local via Ollama)

In [ ]:
response = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages
)

display(Markdown(response.choices[0].message.content))